# 04 - Differential Privacy
Notebook ini menjalankan baseline DP-SGD jika Opacus tersedia, atau fallback ringkas bila belum terpasang.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import torch
from sklearn.model_selection import train_test_split

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models import create_baseline_model
from evaluation import evaluate_identification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
artifact = PROJECT_ROOT / 'data' / 'processed' / 'sequence_bundle.npz'
if not artifact.exists():
    raise FileNotFoundError('sequence_bundle.npz not found. Run 02_preprocessing_pipeline first.')

data = np.load(artifact, allow_pickle=True)
X = data['features'].astype(np.float32)
y_raw = data['labels']
labels_unique = sorted(set(y_raw.tolist()))
label_to_idx = {label: i for i, label in enumerate(labels_unique)}
y = np.array([label_to_idx[val] for val in y_raw], dtype=np.int64)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=64, shuffle=False)

In [2]:
from opacus.validators import ModuleValidator
model = create_baseline_model(input_dim=int(X.shape[2]), hidden_dim=64, num_layers=2, num_classes=int(len(labels_unique)), device=device)
# Fix incompatible modules (nn.LSTM -> DPLSTM)
model = ModuleValidator.fix(model).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

dp_enabled = False
epsilon = None
try:
    from opacus import PrivacyEngine
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=1.1,
        max_grad_norm=1.0,
    )
    dp_enabled = True
    print('Opacus enabled: DP-SGD active')
except Exception as e:
    print(f'Opacus not available, running non-DP fallback: {e}')

for epoch in range(1, 6):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
    print(f'Epoch {epoch:02d} | loss={total_loss/max(len(train_loader),1):.4f}')

if dp_enabled:
    epsilon = privacy_engine.get_epsilon(delta=1e-5)

metrics = evaluate_identification(model, test_loader, device=device)
payload = {
    'dp_enabled': dp_enabled,
    'epsilon': float(epsilon) if epsilon is not None else None,
    'delta': 1e-5 if dp_enabled else None,
    'metrics': metrics,
}
out_path = PROJECT_ROOT / 'outputs' / 'reports' / 'dp_metrics.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
model_path = PROJECT_ROOT / 'outputs' / 'models' / 'dp_lstm.pt'
torch.save({'state_dict': model.state_dict(), 'label_to_idx': label_to_idx}, model_path)
print(f'Saved metrics: {out_path}')
print(f'Saved global DP model: {model_path}')
print(payload)

C:\Users\anang\Downloads\Projek Keamanan Informasi\.venv\Lib\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\anang\AppData\Local\Temp\ipykernel_16336\3938383646.py:33: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Opacus enabled: DP-SGD active


Epoch 01 | loss=3.9344


Epoch 02 | loss=3.9208


Epoch 03 | loss=3.9119


Epoch 04 | loss=3.9048


Epoch 05 | loss=3.8988


Saved metrics: C:\Users\anang\Downloads\Projek Keamanan Informasi\outputs\reports\dp_metrics.json
Saved global DP model: C:\Users\anang\Downloads\Projek Keamanan Informasi\outputs\models\dp_lstm.pt
{'dp_enabled': True, 'epsilon': 0.7705814624811613, 'delta': 1e-05, 'metrics': {'loss': 3.9035427146487764, 'accuracy': 0.02562302562302562, 'precision': 0.0006565394420782276, 'recall': 0.02562302562302562, 'f1': 0.0012802743808903974, 'top3_accuracy': 0.0702000702000702, 'top5_accuracy': 0.11547911547911548, 'confusion_matrix': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 68, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 38, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 63, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [